In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
%run ../0-common/env-config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_races"

In [0]:
races_df = (
        spark.table(f"{catalog_name}.{silver_schema}.races")
        .filter(F.col("batch_id") == v_batch_id)
        )
circuits_df = (
        spark.table(f"{catalog_name}.{silver_schema}.circuits")
        .filter(F.col("batch_id") == v_batch_id)
        )

In [0]:
# Join between races_df and circuits_df on the basis of circuit_id
dim_races_df = (
    races_df.join(
        circuits_df,
        races_df.circuit_id == circuits_df.circuit_id
    )
    .select(
        races_df.season,
        races_df.round,
        races_df.race_date,
        races_df.race_name,
        circuits_df.circuit_name,
        circuits_df.locality,
        circuits_df.country
    )
)

In [0]:
dim_races_df = (
    dim_races_df
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
)

In [0]:
if not spark.catalog.tableExists(target_table):
    (
    dim_races_df.write
    .format('delta')
    .mode("overwrite")
    .saveAsTable(target_table)
    )

else:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, target_table)
    (
        delta_table.alias("t")
        .merge(
            dim_races_df.alias("s"),
            "t.season = s.season AND t.round = s.round"
        )
        .whenMatchedUpdate(
            set={
                "race_date": "s.race_date",
                "race_name": "s.race_name",
                "circuit_name": "s.circuit_name",
                "locality": "s.locality",
                "country": "s.country",
                "updated_at": "s.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
display(spark.table(target_table))